In [5]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AirbnbBigData") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

listings_schema = StructType([
    StructField("listing_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("room_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("minimum_nights", IntegerType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("review_scores_rating", DoubleType(), True)
])

reviews_schema = StructType([
    StructField("listing_id", IntegerType(), True),
    StructField("review_id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("reviewer_id", IntegerType(), True)
])

listings_df = spark.read.csv("../data/cleaned/listings_clean.csv", header=True, schema=listings_schema)
reviews_df = spark.read.csv("../data/cleaned/reviews_clean.csv", header=True, schema=reviews_schema)

review_metrics = reviews_df.groupBy("listing_id").agg(
    count("review_id").alias("total_reviews")
)

enriched_listings = listings_df.join(review_metrics, on="listing_id", how="left")
enriched_listings = enriched_listings.na.fill(0, ["total_reviews"])

print("Aggregated Big Data sample:")
enriched_listings.select("listing_id", "city", "price", "total_reviews").show(5)

final_df = enriched_listings.toPandas()
os.makedirs("../data/cleaned/listings_with_metrics", exist_ok=True)
final_df.to_csv("../data/cleaned/listings_with_metrics/data.csv", index=False)
print("Pipeline finished successfully!")

spark.stop()

Aggregated Big Data sample:
+----------+-----+-----+-------------+
|listing_id| city|price|total_reviews|
+----------+-----+-----+-------------+
|   4082273|Paris| 89.0|            1|
|   4823489|Paris| 60.0|            1|
|   4898654|Paris| 95.0|            2|
|    281420|Paris| 53.0|            2|
|   4797344|Paris| 58.0|            1|
+----------+-----+-----+-------------+
only showing top 5 rows
Pipeline finished successfully!


In [6]:
import pandas as pd

# 1. Load the generated metrics data
analysis_df = pd.read_csv("../data/cleaned/listings_with_metrics/data.csv")

# 2. Generate metrics summary
total_listings = len(analysis_df)
avg_price = analysis_df["price"].mean()
total_reviews = analysis_df["total_reviews"].sum()

print("==================================================")
print("   🏠 AIRBNB RENTAL ECONOMY ANALYTICS BRIEF   ")
print("==================================================")
print(f"Total Unique Listings Processed : {total_listings:,}")
print(f"Average Market Nightly Price   : ${avg_price:.2f}")
print(f"Total Combined Customer Reviews : {int(total_reviews):,}")
print("==================================================\n")

# 3. Present data breakdown by city and room type
print("📋 Top 10 High-Value Market Sectors:")
preview_cols = ["listing_id", "city", "room_type", "price", "total_reviews"]
display(analysis_df[preview_cols].sort_values(by="total_reviews", ascending=False).head(10))

   🏠 AIRBNB RENTAL ECONOMY ANALYTICS BRIEF   
Total Unique Listings Processed : 276,693
Average Market Nightly Price   : $431.30
Total Combined Customer Reviews : 5,354,149

📋 Top 10 High-Value Market Sectors:


,listing_id,city,room_type,price,total_reviews
180230,17222007,Paris,Private room,76.0,891
136712,8637229,Hong Kong,Private room,629.0,828
259923,1249964,Paris,Private room,81.0,796
192405,32011332,Paris,Hotel room,27.0,762
156699,2399029,Rome,Private room,39.0,754
84994,32678719,New York,Hotel room,77.0,753
157427,470817,Rome,Private room,35.0,717
101317,24745583,Paris,Hotel room,36.0,710
29115,865289,Rome,Entire place,26.0,702
233044,162163,Paris,Entire place,96.0,700


In [7]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Load your data
df = pd.read_csv("../data/cleaned/listings_with_metrics/data.csv")

# Create a simple dropdown for cities
city_dropdown = widgets.Dropdown(options=["All"] + sorted(df["city"].unique().tolist()), description='City:')

def update_output(city):
    filtered_df = df if city == "All" else df[df["city"] == city]
    
    # Calculate simple stats
    total_listings = len(filtered_df)
    avg_price = filtered_df["price"].mean()
    
    print(f"Stats for {city}:")
    print(f"Total Listings: {total_listings:,}")
    print(f"Average Price: ${avg_price:.2f}")
    display(filtered_df.head(10))

# Display the interactive widget
widgets.interactive(update_output, city=city_dropdown)

interactive(children=(Dropdown(description='City:', options=('All', 'Bangkok', 'Cape Town', 'Hong Kong', 'Ista…

In [9]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Assuming 'analysis_df' is the dataframe from your previous cells
# If your dataframe has a different name, replace 'analysis_df' with yours
df = analysis_df 

city_dropdown = widgets.Dropdown(options=["All"] + sorted(df["city"].unique().tolist()), description='Select City:')

def update_output(city):
    filtered_df = df if city == "All" else df[df["city"] == city]
    
    # Update Metrics
    print(f"--- Dashboard Results for: {city} ---")
    print(f"Total Listings: {len(filtered_df):,}")
    print(f"Average Price: ${filtered_df['price'].mean():.2f}")
    
    # Add a Visual Component
    plt.figure(figsize=(8, 4))
    filtered_df['room_type'].value_counts().plot(kind='bar', color='steelblue')
    plt.title(f"Listing Distribution by Room Type in {city}")
    plt.xlabel("Room Type")
    plt.ylabel("Count")
    plt.show()
    
    # Display Preview
    display(filtered_df.head(10))

widgets.interactive(update_output, city=city_dropdown)

interactive(children=(Dropdown(description='Select City:', options=('All', 'Bangkok', 'Cape Town', 'Hong Kong'…